# LSTM VAE — Research Notebook

This notebook implements and compares **two** sequence-based VAE models:

1. **LSTM VAE** — vanilla LSTM encoder/decoder, uses final hidden state
2. **LSTM + Attention VAE** — bidirectional LSTM with attention mechanism that learns which frames matter most

Both handle **variable-length reps** natively (no resampling to fixed length).

## Key Difference from Summary Stats VAE
- Summary Stats VAE compresses each rep into 97 statistics → loses temporal ordering
- LSTM VAEs process the rep **frame by frame** → preserves temporal patterns
- Can detect timing issues (bounce at bottom, pause, speed changes)

## 1. Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

AI_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if AI_DIR not in sys.path:
    sys.path.insert(0, AI_DIR)

from Utils.utils.utils import (
    ANGLE_NAMES,
    compute_angle_features_2d,
    smooth_angles,
    find_rep_boundaries,
    extract_rep_angles,
)
from process_landmarks.exercise_config import EXERCISE_CONFIGS, EXERCISE_INDEX, NUM_EXERCISES
from mlp.model import LSTMVAE, LSTMAttentionVAE, vae_loss

print(f"Angle features: {len(ANGLE_NAMES)} → embedding dim: 29 (13 angles + 13 vel + 2 sym + 1 depth)")
print(f"Exercises: {list(EXERCISE_INDEX.keys())}")
print(f"PyTorch version: {torch.__version__}")

## 2. Load Training Data & Build Sequences

In [ ]:
TRAINING_DATA_DIR = os.path.join(AI_DIR, "training_data")

def load_reps_as_sequences(data_dir):
    """Load landmark files, extract reps, build (T, 29) sequences."""
    all_data = []
    
    for exercise_dir in sorted(os.listdir(data_dir)):
        exercise_path = os.path.join(data_dir, exercise_dir)
        if not os.path.isdir(exercise_path):
            continue
        
        exercise_idx = EXERCISE_INDEX.get(exercise_dir)
        if exercise_idx is None:
            continue
        
        if 'squat' in exercise_dir:
            config_key = 'heavy_squat'
        else:
            config_key = 'adaptive'
        exercise_config = EXERCISE_CONFIGS.get(config_key, EXERCISE_CONFIGS['heavy_squat'])
        
        npy_files = [f for f in os.listdir(exercise_path) if f.endswith('.npy')]
        print(f"  {exercise_dir}: {len(npy_files)} files")
        
        for fname in npy_files:
            landmarks = np.load(os.path.join(exercise_path, fname))
            lm = landmarks[:, :, :3] if landmarks.shape[2] > 3 else landmarks
            
            angles = compute_angle_features_2d(lm)
            smooth = smooth_angles(angles)
            
            reps, _ = find_rep_boundaries(smooth, exercise_config)
            reps_data = extract_rep_angles(smooth, reps)
            
            for rep_angles in reps_data:
                T = rep_angles.shape[0]
                velocity = np.diff(rep_angles, axis=0)
                angles_trim = rep_angles[:T - 1]
                knee_sym = (angles_trim[:, 2] - angles_trim[:, 3]).reshape(-1, 1)
                hip_sym = (angles_trim[:, 4] - angles_trim[:, 5]).reshape(-1, 1)
                depth = np.zeros((T - 1, 1), dtype=np.float32)
                
                sequence = np.concatenate([angles_trim, velocity, knee_sym, hip_sym, depth], axis=1)
                
                all_data.append({
                    'sequence': sequence.astype(np.float32),  # (T-1, 29)
                    'angles': rep_angles,
                    'exercise': exercise_dir,
                    'exercise_idx': exercise_idx,
                    'source_file': fname,
                    'n_frames': sequence.shape[0],
                })
        
        count = sum(1 for r in all_data if r['exercise'] == exercise_dir)
        print(f"    → {count} rep sequences")
    
    return all_data

data = load_reps_as_sequences(TRAINING_DATA_DIR)
print(f"\nTotal sequences: {len(data)}")

if data:
    lengths = [d['n_frames'] for d in data]
    print(f"Sequence lengths: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.0f}")
else:
    print("No training data found! Place .npy files in training_data/{exercise}/")

## 3. Prepare Padded Batches

LSTM can process variable-length sequences, but for efficient batching we pad to the max length. The loss is computed only on non-padded frames.

In [ ]:
if data:
    # Pad sequences to max length
    sequences = [torch.tensor(d['sequence']) for d in data]
    lengths = [s.shape[0] for s in sequences]
    padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)  # (N, max_T, 29)
    
    # Create mask for loss computation (ignore padded frames)
    max_len = padded.shape[1]
    mask = torch.zeros(len(data), max_len, dtype=torch.bool)
    for i, l in enumerate(lengths):
        mask[i, :l] = True
    
    # Exercise one-hot
    exercise_indices = [d['exercise_idx'] for d in data]
    exercise_oh = torch.zeros(len(data), NUM_EXERCISES)
    for i, idx in enumerate(exercise_indices):
        exercise_oh[i, idx] = 1.0
    
    print(f"Padded batch shape: {padded.shape}")
    print(f"Mask shape: {mask.shape}")
    print(f"Exercise one-hot shape: {exercise_oh.shape}")
    print(f"Lengths: {sorted(set(lengths))}")

## 4. Training Helper

Masked loss function so padded frames don't affect the model.

In [ ]:
def masked_vae_loss(reconstruction, target, mu, log_var, mask, beta=0.5):
    """VAE loss computed only on non-padded frames."""
    # reconstruction, target: (batch, T, 29)
    # mask: (batch, T) — True for real frames
    
    # Reconstruction loss: MSE only on real frames
    diff = (reconstruction - target) ** 2  # (batch, T, 29)
    mask_expanded = mask.unsqueeze(-1).float()  # (batch, T, 1)
    masked_diff = diff * mask_expanded
    recon_loss = masked_diff.sum() / mask_expanded.sum() / target.shape[-1]
    
    # KL divergence (independent of sequence length)
    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
    
    total = recon_loss + beta * kl_loss
    return total, recon_loss, kl_loss


def train_model(model, padded, exercise_oh, mask, has_attention=False,
                epochs=200, lr=1e-3, beta=0.5, name="Model"):
    """Train an LSTM-based VAE and return loss history."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'total': [], 'recon': [], 'kl': []}
    
    model.train()
    for epoch in range(epochs):
        if has_attention:
            reconstruction, mu, log_var, attn_weights = model(padded, exercise_oh)
        else:
            reconstruction, mu, log_var = model(padded, exercise_oh)
        
        loss, recon_loss, kl_loss = masked_vae_loss(
            reconstruction, padded, mu, log_var, mask, beta
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        history['total'].append(loss.item())
        history['recon'].append(recon_loss.item())
        history['kl'].append(kl_loss.item())
        
        if (epoch + 1) % 20 == 0:
            print(f"  [{name}] Epoch {epoch+1}/{epochs} — loss: {loss.item():.6f} "
                  f"(recon: {recon_loss.item():.6f}, kl: {kl_loss.item():.6f})")
    
    return history

print("Training helpers defined.")

## 5. Train Both Models

### Version A: LSTM VAE (no attention)
Uses final hidden state to summarize the rep. Simpler but may forget early frames in long reps.

### Version B: LSTM + Attention VAE
Uses attention over ALL hidden states. Learns which frames matter most. Bidirectional LSTM sees both past and future context.

In [ ]:
# Hyperparameters
EPOCHS = 200
LR = 1e-3
BETA = 0.5
HIDDEN_DIM = 128
LATENT_DIM = 32

if data:
    # --- Version A: LSTM VAE ---
    print("=" * 50)
    print("Training LSTM VAE (no attention)")
    print("=" * 50)
    lstm_model = LSTMVAE(input_dim=29, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM)
    lstm_history = train_model(
        lstm_model, padded, exercise_oh, mask,
        has_attention=False, epochs=EPOCHS, lr=LR, beta=BETA, name="LSTM"
    )
    
    print()
    
    # --- Version B: LSTM + Attention VAE ---
    print("=" * 50)
    print("Training LSTM + Attention VAE")
    print("=" * 50)
    attn_model = LSTMAttentionVAE(input_dim=29, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM)
    attn_history = train_model(
        attn_model, padded, exercise_oh, mask,
        has_attention=True, epochs=EPOCHS, lr=LR, beta=BETA, name="LSTM+Attn"
    )
    
    print("\nBoth models trained!")
else:
    print("No data available.")

## 6. Compare Training Loss Curves

In [ ]:
if data:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Total loss comparison
    axes[0].plot(lstm_history['total'], label='LSTM VAE', alpha=0.8)
    axes[0].plot(attn_history['total'], label='LSTM+Attention VAE', alpha=0.8)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Total Loss')
    axes[0].legend()
    axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.3)
    
    # Reconstruction loss
    axes[1].plot(lstm_history['recon'], label='LSTM VAE', alpha=0.8)
    axes[1].plot(attn_history['recon'], label='LSTM+Attention VAE', alpha=0.8)
    axes[1].set_xlabel('Epoch')
    axes[1].set_title('Reconstruction Loss')
    axes[1].legend()
    axes[1].set_yscale('log')
    axes[1].grid(True, alpha=0.3)
    
    # KL loss
    axes[2].plot(lstm_history['kl'], label='LSTM VAE', alpha=0.8)
    axes[2].plot(attn_history['kl'], label='LSTM+Attention VAE', alpha=0.8)
    axes[2].set_xlabel('Epoch')
    axes[2].set_title('KL Divergence Loss')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle('LSTM VAE vs LSTM+Attention VAE — Training Comparison', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print(f"Final losses:")
    print(f"  LSTM VAE:      total={lstm_history['total'][-1]:.6f}, recon={lstm_history['recon'][-1]:.6f}")
    print(f"  LSTM+Attn VAE: total={attn_history['total'][-1]:.6f}, recon={attn_history['recon'][-1]:.6f}")

## 7. Evaluate — Reconstruction Errors & Score Comparison

In [ ]:
def sigmoid_calibrate(error, median, iqr):
    scale = max(iqr * 2, 1e-6)
    return 1.0 / (1.0 + np.exp((error - median * 1.5) / scale))

def evaluate_model(model, data, exercise_oh, has_attention=False, name="Model"):
    """Compute per-rep reconstruction errors for an LSTM VAE."""
    model.eval()
    errors = []
    attn_weights_list = []
    
    with torch.no_grad():
        for i, d in enumerate(data):
            x = torch.tensor(d['sequence']).unsqueeze(0)
            e = exercise_oh[i].unsqueeze(0)
            
            if has_attention:
                recon, mu, log_var, attn_w = model(x, e)
                attn_weights_list.append(attn_w.squeeze(0).numpy())
            else:
                recon, mu, log_var = model(x, e)
            
            error = float(((x - recon) ** 2).mean())
            errors.append(error)
    
    errors = np.array(errors)
    median = float(np.median(errors))
    iqr = float(np.percentile(errors, 75) - np.percentile(errors, 25))
    scores = np.array([sigmoid_calibrate(e, median, iqr) for e in errors])
    
    print(f"\n{name} Results:")
    print(f"  Median error: {median:.6f}, IQR: {iqr:.6f}")
    print(f"  Score range: [{scores.min():.3f}, {scores.max():.3f}], mean: {scores.mean():.3f}")
    
    return errors, scores, median, iqr, attn_weights_list

if data:
    lstm_errors, lstm_scores, lstm_median, lstm_iqr, _ = evaluate_model(
        lstm_model, data, exercise_oh, has_attention=False, name="LSTM VAE"
    )
    attn_errors, attn_scores, attn_median, attn_iqr, attn_weights = evaluate_model(
        attn_model, data, exercise_oh, has_attention=True, name="LSTM+Attention VAE"
    )
    
    # Score comparison plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.hist(lstm_scores, bins=15, alpha=0.6, label='LSTM VAE', edgecolor='black')
    ax1.hist(attn_scores, bins=15, alpha=0.6, label='LSTM+Attn VAE', edgecolor='black')
    ax1.set_xlabel('Similarity Score')
    ax1.set_ylabel('Count')
    ax1.set_title('Score Distributions (Training Set)')
    ax1.legend()
    
    ax2.scatter(lstm_scores, attn_scores, alpha=0.6)
    ax2.plot([0, 1], [0, 1], 'r--', alpha=0.5)
    ax2.set_xlabel('LSTM VAE Score')
    ax2.set_ylabel('LSTM+Attention VAE Score')
    ax2.set_title('Score Correlation Between Models')
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()

## 8. Visualize Attention Weights

The attention mechanism learns which frames are most important for assessing form.
High attention on the bottom of the squat or the transition point is a good sign — those are biomechanically critical moments.

In [ ]:
if data and attn_weights:
    n_show = min(4, len(data))
    fig, axes = plt.subplots(n_show, 2, figsize=(14, 3 * n_show))
    if n_show == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_show):
        rep = data[i]
        weights = attn_weights[i]
        angles = rep['angles']
        
        # Left: attention weights over time
        axes[i, 0].bar(range(len(weights)), weights, alpha=0.7, color='orange')
        axes[i, 0].set_ylabel('Attention Weight')
        axes[i, 0].set_title(f"Rep {i+1}: Attention Weights ({rep['exercise']}, {rep['n_frames']} frames)")
        
        # Right: knee angle with attention overlay
        knee_signal = angles[:len(weights), 2]  # left knee
        ax2 = axes[i, 1]
        ax2.plot(knee_signal, 'b-', label='Left Knee Angle')
        ax2_twin = ax2.twinx()
        ax2_twin.fill_between(range(len(weights)), weights, alpha=0.3, color='orange', label='Attention')
        ax2.set_ylabel('Angle (degrees)', color='blue')
        ax2_twin.set_ylabel('Attention', color='orange')
        ax2.set_title(f"Rep {i+1}: Knee Angle + Attention Overlay")
        ax2.legend(loc='upper left')
        ax2_twin.legend(loc='upper right')
    
    axes[-1, 0].set_xlabel('Frame')
    axes[-1, 1].set_xlabel('Frame')
    plt.tight_layout()
    plt.show()
    
    print("Look for attention peaks at the bottom of the squat (lowest knee angle).")
    print("This suggests the model focuses on the most biomechanically critical moment.")
elif data:
    print("No attention weights — attention model may not have been trained.")

## 9. Camera Angle Consistency Test

In [ ]:
if data:
    from collections import defaultdict
    
    file_lstm = defaultdict(list)
    file_attn = defaultdict(list)
    for i, d in enumerate(data):
        file_lstm[d['source_file']].append(lstm_scores[i])
        file_attn[d['source_file']].append(attn_scores[i])
    
    if len(file_lstm) > 1:
        print("Scores by source file (camera angle proxy):\n")
        print(f"{'File':<25} {'LSTM mean':>10} {'LSTM std':>10} {'Attn mean':>10} {'Attn std':>10} {'n':>5}")
        print("-" * 75)
        for fname in sorted(file_lstm.keys()):
            ls = np.array(file_lstm[fname])
            at = np.array(file_attn[fname])
            print(f"{fname[:25]:<25} {ls.mean():>10.3f} {ls.std():>10.3f} {at.mean():>10.3f} {at.std():>10.3f} {len(ls):>5}")
        
        # Variance comparison
        lstm_stds = [np.std(file_lstm[k]) for k in file_lstm if len(file_lstm[k]) > 1]
        attn_stds = [np.std(file_attn[k]) for k in file_attn if len(file_attn[k]) > 1]
        if lstm_stds:
            print(f"\nAvg within-file score std:")
            print(f"  LSTM VAE: {np.mean(lstm_stds):.4f}")
            print(f"  LSTM+Attn VAE: {np.mean(attn_stds):.4f}")
            print("(Lower = more consistent across camera angles)")
    else:
        print("Only one source file — add data from multiple camera angles.")

## 10. Export Best Model

Choose which model performed better and save it for production use.

To activate in production:
- LSTM VAE: set `ACTIVE_MODEL = "lstm_vae"` in `AI/process_landmarks/model_config.py`
- LSTM+Attention VAE: set `ACTIVE_MODEL = "lstm_attention_vae"` in `AI/process_landmarks/model_config.py`

In [ ]:
import json

if data:
    MODELS_DIR = os.path.join(AI_DIR, "mlp", "models")
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    # Compute training avg flexion
    all_flexions = []
    for d in data:
        knee_inner = (d['angles'][:, 2].min() + d['angles'][:, 3].min()) / 2
        all_flexions.append(180.0 - knee_inner)
    training_avg_flexion = float(np.mean(all_flexions))
    
    # Save LSTM VAE
    torch.save(lstm_model.state_dict(), os.path.join(MODELS_DIR, "lstm_vae.pt"))
    print(f"Saved LSTM VAE to {MODELS_DIR}/lstm_vae.pt")
    
    # Save LSTM+Attention VAE
    torch.save(attn_model.state_dict(), os.path.join(MODELS_DIR, "lstm_attention_vae.pt"))
    print(f"Saved LSTM+Attention VAE to {MODELS_DIR}/lstm_attention_vae.pt")
    
    # Save config for whichever model you want to use
    # Change model_type to match your choice
    CHOSEN_MODEL = "lstm_attention_vae"  # or "lstm_vae"
    chosen_median = attn_median if CHOSEN_MODEL == "lstm_attention_vae" else lstm_median
    chosen_iqr = attn_iqr if CHOSEN_MODEL == "lstm_attention_vae" else lstm_iqr
    
    config = {
        "median_error": chosen_median,
        "iqr": chosen_iqr,
        "training_avg_flexion": training_avg_flexion,
        "n_training_reps": len(data),
        "model_type": CHOSEN_MODEL,
        "epochs": EPOCHS,
        "beta": BETA,
        "hidden_dim": HIDDEN_DIM,
        "latent_dim": LATENT_DIM,
    }
    with open(os.path.join(MODELS_DIR, "config.json"), "w") as f:
        json.dump(config, f, indent=2)
    print(f"\nSaved config for {CHOSEN_MODEL}")
    print(f"Calibration: median={chosen_median:.6f}, iqr={chosen_iqr:.6f}")
    print(f"Training avg flexion: {training_avg_flexion:.1f} degrees")
    print(f"\nTo activate: set ACTIVE_MODEL = \"{CHOSEN_MODEL}\" in model_config.py")